# 09 — Runtime retrieval, bounded loops, and cross-references

## Recruiter requirement addressed

This notebook turns the three RAG-loop articles into one executable path: route the question to a single-fact or multi-field contract, retrieve at increasing depths, follow explicit section references, and stop only when the deterministic evidence gate is complete, conflicting, or out of budget.

The deployed Gemini cache is intentionally scoped to the 87 chunks of the main Groupe Foyer QRT. Other documents visibly use the hashing baseline. This is a targeted trained-embedding capability, not a claim that all 1,918 chunks have Gemini vectors.

In [1]:
from pathlib import Path
from IPython.display import display
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table
ROOT = bootstrap()
for module_name in list(sys.modules):
    if module_name == 'app' or module_name.startswith('app.'):
        del sys.modules[module_name]
from app.domain.models import Candidate, EvidenceConstraints, QuestionRequest, ScopeSelection, ScoreTrace, Chunk, SourceLocator
from app.retrieval.references import resolve_references
from app.services.engine import EvidenceEngine
engine = EvidenceEngine()
engine.online_composer.api_key = ''  # isolate retrieval; no generation call in this notebook

## Experiment 1 — early stop versus budget exhaustion

Both requests start at top-1 per field and may expand to top-3 and top-5. The 2025 request should stop as soon as all three QRT fields are accepted. The 2024 request must never reuse 2025 evidence; it expands until the fixed budget is exhausted and returns `NOT_FOUND`.

In [2]:
base = dict(
    mode='deep',
    scope=ScopeSelection(document_ids=['foyer_group_qrt_2025']),
    profile_id='prudential_coverage',
)
complete = await engine.answer(QuestionRequest(
    question="What public evidence describes Groupe Foyer's prudential coverage in 2025?",
    constraints=EvidenceConstraints(entity='Groupe Foyer', period='2025'), **base,
))
wrong_period = await engine.answer(QuestionRequest(
    question="What public evidence describes Groupe Foyer's prudential coverage in 2024?",
    constraints=EvidenceConstraints(entity='Groupe Foyer', period='2024'), **base,
))
rows = [{
    'case': name, 'status': answer.status,
    'strategy': answer.retrieval_run.strategy,
    'dense_provider': answer.retrieval_run.dense_provider,
    'k_history': answer.retrieval_run.k_history,
    'stop_reason': answer.retrieval_run.stop_reason,
    'covered_fields': [item.field_id for item in answer.coverage if item.state == 'COVERED'],
} for name, answer in [('answerable_2025', complete), ('wrong_period_2024', wrong_period)]]
assert complete.status == 'COMPLETE'
assert complete.retrieval_run.dense_provider == 'gemini'
assert complete.retrieval_run.stop_reason == 'contract_complete'
assert wrong_period.status == 'NOT_FOUND'
assert wrong_period.retrieval_run.stop_reason == 'budget_exhausted'
display(display_table(rows))

,case,status,strategy,dense_provider,k_history,stop_reason,covered_fields
0,answerable_2025,COMPLETE,batch_multi_field,gemini,[1],contract_complete,"[eligible_own_funds_scr, group_scr, scr_covera..."
1,wrong_period_2024,NOT_FOUND,batch_multi_field,gemini,"[1, 3, 5]",budget_exhausted,[]


## Experiment 2 — deterministic cross-reference resolution

The resolver recognizes explicit `see section`  patterns, stays inside the selected document, follows at most three references per batch, and exposes unresolved targets. The controlled fixture uses wording found in the public sustainability document while keeping the expected target unambiguous.

In [3]:
def chunk(chunk_id, text, section):
    return Chunk(id=chunk_id, document_id='foyer_sustainability_statement', text=text, locator=SourceLocator(
        document_id='foyer_sustainability_statement', document_title='Foyer Sustainability Statement',
        version='2025', source_url='https://groupe.foyer.lu/', page=1, section_path=[section]))
source = chunk('source-reference', 'For the selection process, see ESRS 2 section 1.6.3.', 'Overview')
target = chunk('target-section', 'Material impacts, risks and opportunities selection process.', 'ESRS 2 1.6.3')
candidate = Candidate(field_id='process', chunk=source, score=ScoreTrace(rrf_score=0.1))
resolved = resolve_references([candidate], [source, target])
missing_source = chunk('missing-reference', 'See section 9.9.', 'Overview')
missing = resolve_references([candidate.model_copy(update={'chunk': missing_source})], [missing_source])
assert [item.chunk.id for item in resolved.candidates] == ['source-reference', 'target-section']
assert resolved.resolved and not resolved.unresolved
assert missing.unresolved and not missing.resolved
display(display_table([
    {'case': 'resolved', 'followed': resolved.resolved, 'unresolved': resolved.unresolved},
    {'case': 'missing target', 'followed': missing.resolved, 'unresolved': missing.unresolved},
]))

,case,followed,unresolved
0,resolved,[source-reference -> section 2 -> target-section],[]
1,missing target,[],[missing-reference -> section 9.9]


## Responsible interpretation

This passes the requested architecture demonstration: trained dense retrieval for the principal QRT, multi-field batch routing, increasing retrieval depth, deterministic stopping, and bounded reference following. It does not claim generic reference understanding: only explicit numbered sections are supported. At scale, the first changes are a persistent vector service, an indexed section registry, asynchronous batches, per-request cost budgets, and evaluation over many real references.